## **Full modular workflow**



### Configuration


In [ ]:
target     = 'ld50'          # output subdirectory name
model_name = 'xgb'           # 'rf', 'xgb', 'svm', 'ridge', 'stacking'

In [ ]:
import os
from pathlib import Path
import sys

BASE_DIR = Path(f"{os.getcwd()}/..").resolve()
sys.path.append(str(BASE_DIR))

os.makedirs(f"{BASE_DIR}/outputs_reg/{target}/models",  exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs_reg/{target}/plots",   exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs_reg/{target}/reports", exist_ok=True)

print(f"BASE_DIR : {BASE_DIR}")
print(f"Target   : {target} | Model: {model_name}")

### Curation

In [ ]:
import pandas as pd
from utils_reg.reg_curation import curate_data

# Load data
df = pd.read_csv(f"{BASE_DIR}/ld50_data/ld50_db.csv")

# Curate data
df_curated = curate_data(df,"SMILES", "LD50")
display(df_curated.head())

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

# Vectorized pLD50 conversion (avoids slow row-by-row loop)
def smiles_to_mw(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Descriptors.ExactMolWt(mol) if mol else np.nan

mw_series = df_curated["SMILES"].apply(smiles_to_mw)

df_curated["pLD50"] = -np.log10(df_curated["LD50"] / (mw_series * 1000))

# Drop rows where MW could not be computed
n_before = len(df_curated)
df_curated = df_curated.dropna(subset=["pLD50"]).reset_index(drop=True)
print(f"Molecules after pLD50 conversion: {len(df_curated)} (dropped {n_before - len(df_curated)})")
print(df_curated[["SMILES", "LD50", "pLD50"]].head())

In [ ]:
from utils_reg.pld50_distribution import plot_pld50_distribution
plot = plot_pld50_distribution(df_curated)
plot.savefig(f"{BASE_DIR}/outputs_reg/plots/pld50_distribution.png")
plot.show()

### Descriptors


In [ ]:
from utils_reg.descriptors import descriptor_matrix

# Generate X and y
X, y = descriptor_matrix(df_curated, "SMILES", "pLD50")

# Full descriptor list for saving model components
full_descriptor_list = X.columns.tolist()

print(f"Descriptor matrix shape: {X.shape}")

### Data split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    shuffle=True,
    test_size=0.3,
    random_state=42
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

### Feature filtering


In [ ]:
from utils_reg.filtering import filter_features

# X_train is a DataFrame here — filtering requires column names
X_train_filtered, X_test_filtered = filter_features(X_train, X_test)

### Preprocessing & Genetic Algorithm

Preprocessing is fitted on all post-filtering features **before** the GA runs.  
This ensures the GA's internal CV evaluates fitness on properly scaled data.  
Pipeline: `filter → scale (fit on all filtered features) → GA (on scaled data) → subset to selected features`

In [ ]:
from utils_reg.preprocessor import build_preprocessor
from utils_reg.robust_ga import ga_feature_selection
from utils_reg.config import RANDOM_STATE, GA_N_GEN, GA_POP_SIZE

# --- Step 1: Preprocessing on ALL filtered features ---
preprocessor = build_preprocessor()
filtered_features = X_train_filtered.columns.tolist()

X_train_scaled = preprocessor.fit_transform(X_train_filtered)
X_test_scaled  = preprocessor.transform(X_test_filtered)

print(f"Preprocessor fitted on {len(filtered_features)} features")

# --- Step 2: Genetic Algorithm on pre-scaled data ---
selected_features, best_fitness, hof, log = ga_feature_selection(
    X_train_scaled, y_train, filtered_features,
    n_gen=GA_N_GEN, pop_size=GA_POP_SIZE, random_state=RANDOM_STATE
)

# --- Step 3: Subset scaled matrices to GA-selected features ---
selected_indices = [filtered_features.index(f) for f in selected_features]
X_train_proc = X_train_scaled[:, selected_indices]
X_test_proc  = X_test_scaled[:, selected_indices]

print(f"\nX_train_proc shape : {X_train_proc.shape}")
print(f"X_test_proc shape  : {X_test_proc.shape}")

# --- Save selected features ---
import pandas as pd
selected_features_df = pd.DataFrame(selected_features, columns=["Selected_Features"])
selected_features_df.to_csv(
    f"{BASE_DIR}/outputs_reg/{target}/reports/selected_features.csv", index=False
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- GA Evolution Plot ---
gen  = log.select("gen")
avgs = log.select("avg")
maxs = log.select("max")

# Fitness = -(RMSE + penalties) → best RMSE ≈ -max_fitness
best_rmse = [-v for v in maxs]
avg_rmse  = [-v for v in avgs]

fig_ga, ax = plt.subplots(figsize=(10, 4))
ax.plot(gen, best_rmse, "b-o", markersize=4, linewidth=2, label="Best RMSE (CV)")
ax.plot(gen, avg_rmse,  "g--",              linewidth=1.5, label="Population avg RMSE")
ax.fill_between(gen, best_rmse, avg_rmse, alpha=0.12, color="royalblue")
ax.set_xlabel("Generation", fontsize=11)
ax.set_ylabel("CV RMSE (pLD50)", fontsize=11)
ax.set_title(
    f"GA Evolution — Feature Selection\n"
    f"Final: {len(selected_features)} features | Best CV RMSE = {-best_fitness:.4f}",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/outputs_reg/{target}/plots/ga_evolution.png", dpi=300)
plt.show()

print(f"\nSelected features ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")

### Training with optimization

In [ ]:
from utils_reg.optimization import optimize_model_regression, train_stacking_model_regression, save_model

if model_name == "stacking":
    final_model = train_stacking_model_regression(X_train_proc, y_train)
else:
    final_model = optimize_model_regression(X_train_proc, y_train, model_name)

save_model(
    BASE_DIR, target, model_name, final_model,
    full_descriptor_list, filtered_features, selected_features, preprocessor
)

### Internal validation

In [ ]:
from utils_reg.validation import compute_metrics_regression

fig_val, metrics = compute_metrics_regression(
    final_model, X_test_proc, y_test, model_name, data_type="internal"
)
fig_val.show()

metrics["metrics_df"].to_csv(
    f"{BASE_DIR}/outputs_reg/{target}/reports/internal_validation_{target}_{model_name}.csv",
    index=False
)
fig_val.savefig(
    f"{BASE_DIR}/outputs_reg/{target}/plots/internal_validation_{target}_{model_name}.png",
    dpi=300, bbox_inches="tight"
)

### Applicability Domain (AD)


In [ ]:
from utils_reg.applicability_domain import applicability_domain_analysis

y_pred_test = metrics["y_pred"]

fig_ad, ad_stats = applicability_domain_analysis(
    model_name, X_train_proc, X_test_proc, y_test, y_pred_test
)
fig_ad.show()
fig_ad.savefig(
    f"{BASE_DIR}/outputs_reg/{target}/plots/ad_{target}_{model_name}.png",
    dpi=300, bbox_inches="tight"
)

print(f"\nAD summary:")
print(f"  h*               = {ad_stats['h_star']:.4f}")
print(f"  Outside AD       = {ad_stats['outside_ad'].sum()} ({ad_stats['pct_outside_ad']:.1f}%)")

### SHAP analysis

In [ ]:
from utils_reg.shap import shap_analysis

fig_shap, shap_dict = shap_analysis(
    model=final_model,
    model_name=model_name,
    X_train_proc=X_train_proc,
    X_test_proc=X_test_proc,
    feature_names=selected_features,
    n_background=100,
    n_explain=500
)

if fig_shap is not None:
    fig_shap.show()
    fig_shap.savefig(
        f"{BASE_DIR}/outputs_reg/{target}/plots/shap_{target}_{model_name}.png",
        dpi=300, bbox_inches="tight"
    )
    print("\nTop 10 features by mean |SHAP|:")
    for i, name in enumerate(shap_dict["top_features"][:10], 1):
        print(f"  {i:2d}. {name}")

### External Validation

In [ ]:
from utils_reg.reg_curation import curate_data
from utils_reg.descriptors import descriptor_matrix

# Load external LD50 dataset
# Expected columns: SMILES, LD50
df_ext = pd.read_csv(f"{BASE_DIR}/ld50_data/ld50_ext.csv")

# Curation
df_ext_curated = curate_data(df_ext, "SMILES", "LD50")

# Vectorized pLD50 conversion
mw_ext = df_ext_curated["SMILES"].apply(smiles_to_mw)
df_ext_curated["pLD50"] = -np.log10(df_ext_curated["LD50"] / (mw_ext * 1000))
df_ext_curated = df_ext_curated.dropna(subset=["pLD50"]).reset_index(drop=True)

# Descriptor calculation
X_ext, y_ext = descriptor_matrix(df_ext_curated, "SMILES", "pLD50")

# Apply preprocessing pipeline:
# 1. Filter to filtered_features (same as training)
# 2. Apply trained preprocessor
# 3. Subset to selected_features
X_ext_filtered = X_ext[filtered_features]
X_ext_scaled   = preprocessor.transform(X_ext_filtered)
X_ext_proc     = X_ext_scaled[:, selected_indices]

# Evaluation
fig_ext, metrics_ext = compute_metrics_regression(
    final_model, X_ext_proc, y_ext, model_name, data_type="external"
)
fig_ext.show()

metrics_ext["metrics_df"].to_csv(
    f"{BASE_DIR}/outputs_reg/{target}/reports/external_validation_{target}_{model_name}.csv",
    index=False
)
fig_ext.savefig(
    f"{BASE_DIR}/outputs_reg/{target}/plots/external_validation_{target}_{model_name}.png",
    dpi=300, bbox_inches="tight"
)

In [ ]:
y_pred_ext = metrics_ext["y_pred"]

fig_ad_ext, ad_stats_ext = applicability_domain_analysis(
    model_name, X_train_proc, X_ext_proc, y_ext, y_pred_ext
)
fig_ad_ext.show()
fig_ad_ext.savefig(
    f"{BASE_DIR}/outputs_reg/{target}/plots/ad_ext_{target}_{model_name}.png",
    dpi=300, bbox_inches="tight"
)

print(f"\nExternal AD summary:")
print(f"  h*            = {ad_stats_ext['h_star']:.4f}")
print(f"  Outside AD    = {ad_stats_ext['outside_ad'].sum()} ({ad_stats_ext['pct_outside_ad']:.1f}%)")